In [4]:
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(MODEL_NAME, device=device)

print("Device:", device)
print("Maximum sequence length:", model.max_seq_length)
print(
    "Embedding dimension:",
    model.get_embedding_dimension()
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Device: cpu
Maximum sequence length: 256
Embedding dimension: 384


In [5]:
sentences = [
    "The application crashes when uploading an image.",
    "Uploading a picture causes the software to close unexpectedly.",
    "The dashboard takes too long to load."
]

embeddings = model.encode(
    sentences,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

Embedding shape: (3, 384)


In [6]:
similarity_matrix = embeddings @ embeddings.T

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=["Report A", "Report B", "Report C"],
    columns=["Report A", "Report B", "Report C"]
)

display(similarity_df.round(4))

,Report A,Report B,Report C
Report A,1.0000,0.7275,0.1867
Report B,0.7275,1.0000,0.2033
Report C,0.1867,0.2033,1.0000


In [7]:
test_reports = [
    "The application crashes during login.",
    " ".join(["software error"] * 300)
]

token_lengths = [
    len(
        model.tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )
    for text in test_reports
]

for i, length in enumerate(token_lengths, start=1):
    print(f"Report {i}: {length} tokens")

print("Model limit:", model.max_seq_length)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (602 > 256). Running this sequence through the model will result in indexing errors


Report 1: 9 tokens
Report 2: 602 tokens
Model limit: 256


In [8]:
issuedupes = pd.read_csv(
    processed_dir / "issuedupes_issues_grouped.csv",
    low_memory=False
)

eclipse = pd.read_csv(
    processed_dir / "eclipse_issues_grouped.csv",
    low_memory=False
)

datasets = {}

for name, df in [
    ("IssueDupesLite", issuedupes),
    ("Eclipse Platform", eclipse)
]:
    df = df.copy()

    df["issue_id"] = df["issue_id"].astype(str)
    df["created_at"] = pd.to_datetime(
        df["created_at"],
        utc=True,
        errors="coerce"
    )
    df["text"] = df["text"].fillna("").astype(str)

    df = df[
        df["text"].str.strip().ne("")
        & df["created_at"].notna()
    ].sort_values(
        ["created_at", "issue_id"]
    ).reset_index(drop=True)

    datasets[name] = df

    print(f"{name}: {len(df):,} reports")

IssueDupesLite: 19,094 reports
Eclipse Platform: 85,155 reports


In [9]:
token_stats = {}

for name, df in datasets.items():

    print(f"\nAnalyzing {name}...")

    sample = df["text"].sample(
        n=min(2000, len(df)),
        random_state=42
    )

    encoded = model.tokenizer(
        sample.tolist(),
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_attention_mask=False
    )

    lengths = np.array([
        len(ids) for ids in encoded["input_ids"]
    ])

    token_stats[name] = {
        "Sample size": len(lengths),
        "Mean tokens": round(lengths.mean(), 2),
        "Median tokens": int(np.median(lengths)),
        "95th percentile": round(
            np.percentile(lengths, 95), 2
        ),
        "Max tokens": int(lengths.max()),
        "Over 256 tokens (%)": round(
            100 * np.mean(lengths > model.max_seq_length),
            2
        )
    }

token_summary = pd.DataFrame(token_stats).T

display(token_summary)


Analyzing IssueDupesLite...

Analyzing Eclipse Platform...


,Sample size,Mean tokens,Median tokens,95th percentile,Max tokens,Over 256 tokens (%)
IssueDupesLite,2000.0,434.60,159.0,1540.45,30427.0,33.30
Eclipse Platform,2000.0,384.95,121.0,1547.25,15958.0,23.45


In [10]:
import json
from tqdm.auto import tqdm

embeddings_dir = project_root / "models" / "embeddings"
embeddings_dir.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 16

print("Embedding directory:", embeddings_dir)
print("Batch size:", BATCH_SIZE)


Embedding directory: c:\Users\user\Documents\fisayo\duplicate-bug-report-detection\models\embeddings
Batch size: 16


In [11]:
def generate_embeddings(name, df, model, batch_size=16):
    dataset_name = name.lower().replace(" ", "_")

    embedding_path = embeddings_dir / f"{dataset_name}_embeddings.npy"
    ids_path = embeddings_dir / f"{dataset_name}_issue_ids.csv"
    metadata_path = embeddings_dir / f"{dataset_name}_metadata.json"

    texts = df["text"].tolist()
    issue_ids = df["issue_id"].astype(str).tolist()

    dimension = model.get_sentence_embedding_dimension()

    embeddings = np.lib.format.open_memmap(
        embedding_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(texts), dimension)
    )

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc=f"Encoding {name}"
    ):
        end = min(start + batch_size, len(texts))

        batch_embeddings = model.encode(
            texts[start:end],
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )

        embeddings[start:end] = batch_embeddings

    embeddings.flush()
    del embeddings

    pd.DataFrame({
        "issue_id": issue_ids
    }).to_csv(ids_path, index=False)

    metadata = {
        "dataset": name,
        "model": MODEL_NAME,
        "max_sequence_length": model.max_seq_length,
        "normalized": True,
        "embedding_dimension": dimension,
        "number_of_issues": len(texts)
    }

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"\nCompleted {name}")
    print("Reports encoded:", len(texts))
    print("Embeddings saved:", embedding_path)

    return embedding_path


In [13]:
issuedupes_embedding_path = generate_embeddings(
    name="IssueDupesLite",
    df=datasets["IssueDupesLite"],
    model=model,
    batch_size=BATCH_SIZE
)

C:\Users\user\AppData\Local\Temp\ipykernel_33380\35275612.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = model.get_sentence_embedding_dimension()


Encoding IssueDupesLite:   0%|          | 0/1194 [00:00<?, ?it/s]


Completed IssueDupesLite
Reports encoded: 19094
Embeddings saved: c:\Users\user\Documents\fisayo\duplicate-bug-report-detection\models\embeddings\issuedupeslite_embeddings.npy


In [14]:
issuedupes_embeddings = np.load(
    issuedupes_embedding_path,
    mmap_mode="r"
)

print("Embedding shape:", issuedupes_embeddings.shape)
print("Data type:", issuedupes_embeddings.dtype)

print(
    "All values finite:",
    np.isfinite(issuedupes_embeddings).all()
)

print(
    "First embedding norm:",
    np.linalg.norm(issuedupes_embeddings[0])
)

Embedding shape: (19094, 384)
Data type: float32
All values finite: True
First embedding norm: 1.0


In [15]:
eclipse_embedding_path = generate_embeddings(
    name="Eclipse Platform",
    df=datasets["Eclipse Platform"],
    model=model,
    batch_size=BATCH_SIZE
)

C:\Users\user\AppData\Local\Temp\ipykernel_33380\35275612.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = model.get_sentence_embedding_dimension()


Encoding Eclipse Platform:   0%|          | 0/5323 [00:00<?, ?it/s]


Completed Eclipse Platform
Reports encoded: 85155
Embeddings saved: c:\Users\user\Documents\fisayo\duplicate-bug-report-detection\models\embeddings\eclipse_platform_embeddings.npy


In [16]:
embedding_files = {
    "IssueDupesLite": "issuedupeslite",
    "Eclipse Platform": "eclipse_platform"
}

links_files = {
    "IssueDupesLite": "issuedupes_direct_links.csv",
    "Eclipse Platform": "eclipse_direct_links.csv"
}

embeddings_data = {}
links_data = {}

for name, prefix in embedding_files.items():

    embeddings = np.load(
        embeddings_dir / f"{prefix}_embeddings.npy",
        mmap_mode="r"
    )

    saved_ids = pd.read_csv(
        embeddings_dir / f"{prefix}_issue_ids.csv",
        dtype={"issue_id": str}
    )["issue_id"].tolist()

    current_ids = (
        datasets[name]["issue_id"].astype(str).tolist()
    )

    assert saved_ids == current_ids, (
        f"Issue ordering mismatch for {name}"
    )

    assert embeddings.shape == (
        len(current_ids), 384
    )

    embeddings_data[name] = embeddings

    links = pd.read_csv(
        processed_dir / links_files[name],
        dtype={"source_id": str, "target_id": str}
    )

    links_data[name] = links

    print(f"{name}: {embeddings.shape}")
    print("Issue ordering verified")

IssueDupesLite: (19094, 384)
Issue ordering verified
Eclipse Platform: (85155, 384)
Issue ordering verified


In [17]:
def prepare_semantic_benchmark(issues, links, train_fraction=0.8):
    issues = issues.copy()

    issues["created_at"] = pd.to_datetime(
        issues["created_at"], utc=True
    )

    issues = issues.sort_values(
        ["created_at", "issue_id"]
    ).reset_index(drop=True)

    split_idx = int(len(issues) * train_fraction)
    cutoff_time = issues.iloc[split_idx]["created_at"]

    train = issues[
        issues["created_at"] < cutoff_time
    ]

    test = issues[
        issues["created_at"] >= cutoff_time
    ]

    valid_ids = set(issues["issue_id"])
    adjacency = {}

    for source, target in links[
        ["source_id", "target_id"]
    ].itertuples(index=False, name=None):

        if (
            source not in valid_ids
            or target not in valid_ids
            or source == target
        ):
            continue

        adjacency.setdefault(source, set()).add(target)
        adjacency.setdefault(target, set()).add(source)

    created_lookup = dict(
        zip(issues["issue_id"], issues["created_at"])
    )

    relevant = {}

    for query_id, query_time in zip(
        test["issue_id"], test["created_at"]
    ):
        earlier = {
            other
            for other in adjacency.get(query_id, set())
            if created_lookup[other] < query_time
        }

        if earlier:
            relevant[query_id] = earlier

    eligible = test[
        test["issue_id"].isin(relevant)
    ].copy()

    return {
        "all_issues": issues,
        "train": train,
        "test": test,
        "eligible": eligible,
        "relevant": relevant
    }


semantic_benchmarks = {}

for name in datasets:
    semantic_benchmarks[name] = prepare_semantic_benchmark(
        datasets[name],
        links_data[name]
    )

    benchmark = semantic_benchmarks[name]

    print(f"\n{name}")
    print("Training issues:", len(benchmark["train"]))
    print("Test issues:", len(benchmark["test"]))
    print("Eligible queries:", len(benchmark["eligible"]))


IssueDupesLite
Training issues: 15275
Test issues: 3819
Eligible queries: 2258

Eclipse Platform
Training issues: 68124
Test issues: 17031
Eligible queries: 2174


In [19]:
from tqdm.auto import tqdm

def evaluate_sentence_bert(benchmark, embeddings):

    issues = benchmark["all_issues"]
    eligible = benchmark["eligible"]
    relevant = benchmark["relevant"]

    issue_ids = issues["issue_id"].to_numpy()
    timestamps = issues["created_at"].to_numpy(
        dtype="datetime64[ns]"
    )

    id_to_idx = {
        issue_id: idx
        for idx, issue_id in enumerate(issue_ids)
    }

    ranks = []

    for query in tqdm(
        eligible.itertuples(index=False),
        total=len(eligible),
        desc="Evaluating retrieval"
    ):

        query_idx = id_to_idx[query.issue_id]

        cutoff = np.searchsorted(
            timestamps,
            np.datetime64(
                query.created_at.to_datetime64()
            ),
            side="left"
        )

        scores = np.asarray(
            embeddings[:cutoff] @ embeddings[query_idx]
        )

        relevant_indices = sorted(
            id_to_idx[rid]
            for rid in relevant[query.issue_id]
            if rid in id_to_idx
            and id_to_idx[rid] < cutoff
        )

        if not relevant_indices:
            raise ValueError(
                f"No valid relevant issue for {query.issue_id}"
            )

        best_score = max(
            scores[idx] for idx in relevant_indices
        )

        best_relevant_idx = min(
            idx for idx in relevant_indices
            if scores[idx] == best_score
        )

        first_rank = (
            np.count_nonzero(scores > best_score)
            + np.count_nonzero(
                scores[:best_relevant_idx] == best_score
            )
            + 1
        )

        ranks.append(first_rank)

    ranks = np.asarray(ranks)

    return {
        "Queries": len(ranks),
        "Recall@1": np.mean(ranks <= 1),
        "Recall@5": np.mean(ranks <= 5),
        "Recall@10": np.mean(ranks <= 10),
        "MRR": np.mean(1.0 / ranks)
    }

In [20]:
issuedupes_results = evaluate_sentence_bert(
    semantic_benchmarks["IssueDupesLite"],
    embeddings_data["IssueDupesLite"]
)

print("IssueDupesLite Sentence-BERT results:")

display(
    pd.DataFrame(
        [issuedupes_results],
        index=["IssueDupesLite"]
    ).round(4)
)

Evaluating retrieval:   0%|          | 0/2258 [00:00<?, ?it/s]

IssueDupesLite Sentence-BERT results:


,Queries,Recall@1,Recall@5,Recall@10,MRR
IssueDupesLite,2258,0.6236,0.8047,0.8525,0.7047


In [21]:
eclipse_results = evaluate_sentence_bert(
    semantic_benchmarks["Eclipse Platform"],
    embeddings_data["Eclipse Platform"]
)

print("Eclipse Platform Sentence-BERT results:")

display(
    pd.DataFrame(
        [eclipse_results],
        index=["Eclipse Platform"]
    ).round(4)
)

Evaluating retrieval:   0%|          | 0/2174 [00:00<?, ?it/s]

Eclipse Platform Sentence-BERT results:


,Queries,Recall@1,Recall@5,Recall@10,MRR
Eclipse Platform,2174,0.2065,0.3648,0.4365,0.2836


In [22]:
sentence_bert_results = pd.DataFrame({
    "IssueDupesLite": issuedupes_results,
    "Eclipse Platform": eclipse_results
}).T

results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

sentence_bert_results.index.name = "Dataset"

sentence_bert_results.to_csv(
    results_dir / "sentence_bert_baseline_results.csv"
)

display(sentence_bert_results.round(4))

,Queries,Recall@1,Recall@5,Recall@10,MRR
Dataset,,,,,
IssueDupesLite,2258.0,0.6236,0.8047,0.8525,0.7047
Eclipse Platform,2174.0,0.2065,0.3648,0.4365,0.2836
